# Training: spatial-output topo model (`BioLeakyRNNTopo` + `CuedTargetSpatialV3`)

Mirrors `01c_train_topo.ipynb` 3-stage curriculum (stage 0 → 1 → 2) but with the
new spatial action space: the network's primary output head produces continuous
(x, y) saccade endpoint at every timestep instead of binary hold/release.

Checkpoints: `../checkpoints/stage{0,1,2}_topo_spatial.pt`.


In [ ]:
import sys, os, pathlib
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.model_topo import BioLeakyRNNTopo
from src.env import CuedTargetSpatialV3
from src.train_spatial import SpatialTrainConfig, train_supervised_spatial

device = "cpu"
print("device:", device)
Path("checkpoints").mkdir(exist_ok=True)


In [ ]:
def make_model():
    return BioLeakyRNNTopo(
        input_size=7,
        hidden_size=180,
        output_size=4,  # multi-task: [:2]=(x,y) position, [2:]=hold/release logits
        dt=20.0,
        tau=100.0,
        activation="softplus",
        sigma_rec=0.10,
        rec_init="diag",
        use_ei=True,
        exc_ratio=0.80,
        use_dale=True,
        mask_seed=42,
        sheet_side=12,
        tau_ee=0.25,
        tau_ie=0.32,
        tau_ei=0.64,
        tau_ii=0.64,
        rf_sigma=0.3,
        tau_e_range=(50, 150),
        tau_i_range=(10, 50),
        use_adaptation=False,
    ).to(device)


## Stage 0 — detect target, no cue, no distractors

In [ ]:
def make_env_stage0_spatial():
    return CuedTargetSpatialV3(
        dt=20,
        cue_strength=0.0,
        p_distractor_trial=0.0,
        distractor_strength=0.0,
        continuous_locations=True,
    )


model = make_model()
cfg0 = SpatialTrainConfig(
    batch_size=64,
    lr=1e-3,
    max_updates=1000,
    print_every=100,
    device=device,
)
history0 = train_supervised_spatial(model, make_env_stage0_spatial, cfg0)
torch.save({"state_dict": model.state_dict()}, "checkpoints/stage0_topo_spatial.pt")
print("Saved: checkpoints/stage0_topo_spatial.pt")


## Stage 1 — add cue, no distractors

In [ ]:
def make_env_stage1_spatial():
    return CuedTargetSpatialV3(
        dt=20,
        cue_strength=1.0,
        p_distractor_trial=0.0,
        distractor_strength=0.0,
        continuous_locations=True,
    )


model = make_model()
model.load_state_dict(
    torch.load("checkpoints/stage0_topo_spatial.pt", weights_only=True)["state_dict"],
    strict=False,
)
cfg1 = SpatialTrainConfig(
    batch_size=64,
    lr=1e-3,
    max_updates=1000,
    print_every=100,
    device=device,
)
history1 = train_supervised_spatial(model, make_env_stage1_spatial, cfg1)
torch.save({"state_dict": model.state_dict()}, "checkpoints/stage1_topo_spatial.pt")
print("Saved: checkpoints/stage1_topo_spatial.pt")


## Stage 2 — cue + distractors (full task)

In [ ]:
def make_env_stage2_spatial():
    return CuedTargetSpatialV3(
        dt=20,
        cue_strength=1.0,
        p_distractor_trial=0.6,
        distractor_strength=1.0,
        continuous_locations=True,
    )


model = make_model()
model.load_state_dict(
    torch.load("checkpoints/stage1_topo_spatial.pt", weights_only=True)["state_dict"],
    strict=False,
)
cfg2 = SpatialTrainConfig(
    batch_size=64,
    lr=1e-3,
    max_updates=5000,
    print_every=50,
    device=device,
)
history2 = train_supervised_spatial(model, make_env_stage2_spatial, cfg2)
torch.save({"state_dict": model.state_dict()}, "checkpoints/stage2_topo_spatial.pt")
print("Saved: checkpoints/stage2_topo_spatial.pt")


## Training curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex='col')
for col, (hist, name) in enumerate(zip(
    [history0, history1, history2], ["stage 0", "stage 1", "stage 2"])):
    u = hist["update"]
    # row 0: training MSE + mean response-window err
    ax = axes[0, col]
    ax.plot(u, hist["loss"], label="train MSE", color="steelblue")
    ax.plot(u, hist["mean_err"], label="eval mean_err", color="firebrick", alpha=0.8)
    ax.set_title(name); ax.set_ylabel("loss / err"); ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    # row 1: hit / miss / FA over training
    ax = axes[1, col]
    ax.plot(u, [v*100 for v in hist["p_hit"]],  label="hit",  color="green")
    ax.plot(u, [v*100 for v in hist["p_miss"]], label="miss", color="orange")
    ax.plot(u, [v*100 for v in hist["p_fa"]],   label="FA",   color="red")
    ax.set_xlabel("update"); ax.set_ylabel("%"); ax.set_ylim(-2, 102)
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Quick eval on stage 2

In [ ]:
from src.analysis import collect_trials_spatial
from collections import Counter

trials = collect_trials_spatial(model, make_env_stage2_spatial, n_trials=500, device=device)
out_counts = Counter(tr["train_outcome"] for tr in trials)
total = len(trials)
print(f"Total trials: {total}")
for o, n in sorted(out_counts.items(), key=lambda x: -x[1]):
    print(f"  {o}: {n} ({100*n/total:.1f}%)")

mean_errs = [tr["mean_err"] for tr in trials if tr["mean_err"] is not None]
if mean_errs:
    print(f"\nMean response-window err across all trials: {np.mean(mean_errs):.4f}")
